# 🇨🇦 CAD/USD Exchange‑Rate Alert

This notebook monitors **five independent indicators** that collectively signal
whether CAD is likely to strengthen against USD, then sends an HTML summary
email via **SendGrid**.

| # | Indicator | Data source | Bullish‑for‑CAD condition |
|---|-----------|-------------|--------------------------|
| A | CAD/USD mid‑market rate | Bank of Canada Valet API | Spot > 30-day avg by ≥ +0.5 % |
| B | USD Index (DXY) | Yahoo Finance `DX-Y.NYB` | 5-day linear slope < 0 |
| C | WTI Crude Oil | Yahoo Finance `CL=F` | 5-day linear slope > 0 |
| D | US–CA 10-yr yield spread | Yahoo Finance `^TNX`, `CA10YT=RR` | Spread < 0.5 % |
| E | Market volatility (VIX) | Yahoo Finance `^VIX` | VIX < 18 (calm regime) |

## Quick start
1. Copy `.env.example` → `.env` and fill in your values.
2. `pip install -r requirements.txt`
3. **Run All Cells** (Kernel → Restart & Run All).


In [ ]:
# Uncomment and run once to install all dependencies
# %pip install -r requirements.txt


In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

load_dotenv()
print("✅ Libraries loaded")


In [ ]:
# ── SendGrid ──────────────────────────────────────────────────────────────────
SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY", "")
FROM_EMAIL       = os.getenv("FROM_EMAIL", "alerts@example.com")
TO_EMAIL         = os.getenv("TO_EMAIL", "recipient@example.com")
EMAIL_SUBJECT    = os.getenv(
    "EMAIL_SUBJECT",
    "CAD/USD Alert – " + datetime.now(timezone.utc).strftime("%Y-%m-%d"),
)

# ── Signal thresholds (edit as desired) ───────────────────────────────────────
CAD_STRONG_THRESHOLD = 0.005   # spot must exceed 30-day avg by ≥ 0.5 %
DXY_SLOPE_WINDOW     = 5       # trading days used for DXY slope
WTI_SLOPE_WINDOW     = 5       # trading days used for WTI slope
VIX_CALM_THRESHOLD   = 18      # VIX below this → calm / risk-on regime

print("Configuration loaded:")
print(f"  FROM: {FROM_EMAIL}  →  TO: {TO_EMAIL}")
print(f"  SendGrid key present: {bool(SENDGRID_API_KEY)}")


In [ ]:
# ── A: CAD/USD mid-market rate (Bank of Canada Valet API) ─────────────────────

def fetch_cad_usd(days: int = 92) -> pd.DataFrame:
    """
    Fetch CAD/USD daily mid-market rate from the Bank of Canada Valet API.

    FXCADUSD = USD per 1 CAD  (higher value → stronger CAD).
    Returns a DataFrame with column 'rate', indexed by date.

    Parameters
    ----------
    days : int
        Calendar days of history to request.  92 days ≈ 90 trading days once
        weekends and public holidays are filtered by the API.
    """
    start_date = (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%d")
    url = (
        "https://www.bankofcanada.ca/valet/observations/FXCADUSD/json"
        f"?start_date={start_date}"
    )
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    observations = response.json().get("observations", [])
    records = [
        {
            "date": pd.to_datetime(obs["d"]),
            "rate": float(obs["FXCADUSD"]["v"]),
        }
        for obs in observations
        if obs.get("FXCADUSD", {}).get("v") not in (None, "")
    ]
    return pd.DataFrame(records).set_index("date").sort_index()


def analyze_cad_usd(df: pd.DataFrame) -> dict:
    """Compute spot rate, moving averages, and signal."""
    spot     = float(df["rate"].iloc[-1])
    avg_30   = float(df["rate"].iloc[-30:].mean())
    avg_90   = float(df["rate"].mean())          # caller provides ~90 days
    pct_diff = (spot - avg_30) / avg_30          # positive → CAD above 30d avg
    bullish  = pct_diff >= CAD_STRONG_THRESHOLD
    return {
        "spot":          round(spot, 6),
        "avg_30d":       round(avg_30, 6),
        "avg_90d":       round(avg_90, 6),
        "pct_above_30d": round(pct_diff * 100, 3),  # in percent
        "bullish":       bullish,
        "signal":        "CAD STRONG (spot > 30d avg + 0.5%)" if bullish else "NEUTRAL / BEARISH",
        "icon":          "🟢" if bullish else "🔴",
    }


In [ ]:
# ── B: USD Index (DXY) ────────────────────────────────────────────────────────

def analyze_dxy(window: int = DXY_SLOPE_WINDOW) -> dict:
    """
    Fetch DXY price history and compute a linear slope over the last `window`
    trading days.  A negative slope (DXY falling) → USD weakening → bullish CAD.
    """
    hist = pd.Series(dtype=float)
    for ticker_id in ("DX-Y.NYB", "DX=F"):
        raw = yf.Ticker(ticker_id).history(period="30d")
        if not raw.empty:
            hist = raw["Close"].dropna()
            break

    if hist.empty:
        return {"current": None, "5d_slope": None, "bullish": None,
                "signal": "DATA UNAVAILABLE", "icon": "⚠️"}

    current = float(hist.iloc[-1])
    recent  = hist.iloc[-window:].values
    slope   = float(np.polyfit(range(len(recent)), recent, 1)[0])
    bullish = slope < 0
    return {
        "current":  round(current, 3),
        "5d_slope": round(slope, 4),
        "bullish":  bullish,
        "signal":   "USD WEAKENING" if bullish else "USD STRENGTHENING",
        "icon":     "🟢" if bullish else "🔴",
    }


In [ ]:
# ── C: WTI Crude Oil ──────────────────────────────────────────────────────────

def analyze_wti(window: int = WTI_SLOPE_WINDOW) -> dict:
    """
    Fetch WTI front-month futures and compute a linear slope over `window` days.
    A positive slope (oil rising) → bullish for CAD.
    """
    raw  = yf.Ticker("CL=F").history(period="30d")
    hist = raw["Close"].dropna()

    if hist.empty:
        return {"current": None, "5d_slope": None, "bullish": None,
                "signal": "DATA UNAVAILABLE", "icon": "⚠️"}

    current = float(hist.iloc[-1])
    recent  = hist.iloc[-window:].values
    slope   = float(np.polyfit(range(len(recent)), recent, 1)[0])
    bullish = slope > 0
    return {
        "current":  round(current, 2),
        "5d_slope": round(slope, 4),
        "bullish":  bullish,
        "signal":   "OIL RISING (CAD POSITIVE)" if bullish else "OIL FALLING (CAD NEGATIVE)",
        "icon":     "🟢" if bullish else "🔴",
    }


In [ ]:
# ── D: Interest-rate expectations (US – CA 10-yr yield spread) ───────────────

def analyze_yield_spread() -> dict:
    """
    Proxy for Fed vs BoC rate expectations: US 10-Year minus CA 10-Year
    government bond yield spread.

    A narrow spread (< 0.5 pp) signals more Fed cuts relative to BoC → CAD bullish.

    Tickers
    -------
    ^TNX      – US 10-Year Treasury yield (Yahoo Finance)
    CA10YT=RR – Government of Canada 10-Year benchmark yield (Reuters / Yahoo Finance)
    """

    def _last_close(ticker_id: str):
        raw    = yf.Ticker(ticker_id).history(period="30d")
        closes = raw["Close"].dropna()
        return float(closes.iloc[-1]) if not closes.empty else None

    us_yield = _last_close("^TNX")
    ca_yield = _last_close("CA10YT=RR")

    if us_yield is not None and ca_yield is not None:
        spread  = round(us_yield - ca_yield, 3)
        bullish = spread < 0.5      # narrow → more Fed cuts priced in
        signal  = ("MORE FED CUTS / NARROW SPREAD"
                   if bullish else "FED HAWKISH / WIDE SPREAD")
        icon    = "🟢" if bullish else "🔴"
    else:
        spread  = None
        bullish = None
        signal  = "DATA UNAVAILABLE"
        icon    = "⚠️"

    return {
        "us_10y":  us_yield,
        "ca_10y":  ca_yield,
        "spread":  spread,
        "bullish": bullish,
        "signal":  signal,
        "icon":    icon,
    }


In [ ]:
# ── E: Market Volatility (VIX) ────────────────────────────────────────────────

def analyze_vix() -> dict:
    """
    Fetch the CBOE Volatility Index (VIX).
    VIX < 18 → calm / risk-on regime → supports CAD and other risk currencies.
    """
    raw  = yf.Ticker("^VIX").history(period="5d")
    hist = raw["Close"].dropna()

    if hist.empty:
        return {"current": None, "bullish": None,
                "signal": "DATA UNAVAILABLE", "icon": "⚠️"}

    current = round(float(hist.iloc[-1]), 2)
    bullish = current < VIX_CALM_THRESHOLD
    return {
        "current": current,
        "bullish": bullish,
        "signal":  (f"CALM REGIME (VIX < {VIX_CALM_THRESHOLD})"
                    if bullish else f"HIGH FEAR (VIX >= {VIX_CALM_THRESHOLD})"),
        "icon":    "🟢" if bullish else "🔴",
    }


In [ ]:
# ── HTML email report ─────────────────────────────────────────────────────────

def _signal_bg(bullish) -> str:
    """Map a bullish flag to a table-row background colour."""
    if bullish is True:  return "#d4edda"   # green
    if bullish is False: return "#f8d7da"   # red
    return "#fff3cd"                         # amber (unknown / N/A)


def _td_row(title: str, subtitle: str, value_str: str,
            detail_str: str, icon: str, signal: str, bullish) -> str:
    """Build one <tr> for the indicator summary table."""
    bg = _signal_bg(bullish)
    return (
        f'<tr style="background:{bg};">'
        f'<td><strong>{title}</strong><br/>'
        f'<small style="color:#666;">{subtitle}</small></td>'
        f'<td style="text-align:right;white-space:nowrap;">'
        f'<strong>{value_str}</strong>'
        + (f'<br/><small style="color:#666;">{detail_str}</small>' if detail_str else "")
        + f'</td>'
        f'<td>{icon}&nbsp;{signal}</td>'
        f'</tr>'
    )


def generate_html_report(cad: dict, dxy: dict, wti: dict,
                          yld: dict, vix: dict) -> str:
    """Assemble a self-contained HTML email from indicator dicts."""
    now    = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    flags  = [cad["bullish"], dxy["bullish"], wti["bullish"],
               yld["bullish"], vix["bullish"]]
    n_bull  = sum(1 for f in flags if f is True)
    n_valid = sum(1 for f in flags if f is not None)

    if n_bull >= 4:
        overall_msg = "STRONG SIGNAL &#x2013; multiple indicators align for CAD strength"
        overall_bg  = "#d4edda"
        overall_icon = "&#x1F7E2;"
    elif n_bull >= 3:
        overall_msg = "MODERATE SIGNAL &#x2013; majority of indicators favour CAD strength"
        overall_bg  = "#fff3cd"
        overall_icon = "&#x1F7E1;"
    else:
        overall_msg = "WEAK / NO SIGNAL &#x2013; indicators do not currently favour CAD"
        overall_bg  = "#f8d7da"
        overall_icon = "&#x1F534;"

    # pre-format optional/nullable values
    us_y  = ("%.3f%%" % yld["us_10y"])  if yld["us_10y"]  is not None else "N/A"
    ca_y  = ("%.3f%%" % yld["ca_10y"])  if yld["ca_10y"]  is not None else "N/A"
    spr   = ("%+.3f%%" % yld["spread"]) if yld["spread"]  is not None else "N/A"
    dxy_c = str(dxy["current"])          if dxy["current"] is not None else "N/A"
    dxy_s = ("%+.4f" % dxy["5d_slope"]) if dxy["5d_slope"] is not None else "N/A"
    wti_c = ("$%.2f" % wti["current"])  if wti["current"] is not None else "N/A"
    wti_s = ("%+.4f" % wti["5d_slope"]) if wti["5d_slope"] is not None else "N/A"
    pct30 = "%+.3f%%" % cad["pct_above_30d"]

    row_a = _td_row(
        "A &#8212; CAD/USD (BoC)",
        "30d avg: %s &nbsp;|&nbsp; 90d avg: %s" % (cad["avg_30d"], cad["avg_90d"]),
        str(cad["spot"]), pct30 + " vs 30d avg",
        cad["icon"], cad["signal"], cad["bullish"],
    )
    row_b = _td_row(
        "B &#8212; USD Index (DXY)",
        "5&#8209;day linear slope",
        dxy_c, "Slope: " + dxy_s,
        dxy["icon"], dxy["signal"], dxy["bullish"],
    )
    row_c = _td_row(
        "C &#8212; WTI Crude Oil",
        "5&#8209;day linear slope",
        wti_c, "Slope: " + wti_s,
        wti["icon"], wti["signal"], wti["bullish"],
    )
    row_d = _td_row(
        "D &#8212; Interest&#8209;Rate Spread",
        "US 10Y &#8722; CA 10Y government bond yields",
        "US: " + us_y + " &nbsp;/&nbsp; CA: " + ca_y,
        "Spread: " + spr,
        yld["icon"], yld["signal"], yld["bullish"],
    )
    row_e = _td_row(
        "E &#8212; VIX (Market Volatility)",
        "Calm regime: VIX &lt; " + str(VIX_CALM_THRESHOLD),
        str(vix["current"]), "",
        vix["icon"], vix["signal"], vix["bullish"],
    )

    score_txt = "<strong>Score: %d/%d indicators bullish for CAD</strong>" % (n_bull, n_valid)

    parts = [
        "<!DOCTYPE html>",
        '<html lang="en">',
        "<head>",
        '  <meta charset="UTF-8"/>',
        "  <style>",
        "    body  { font-family: Arial, sans-serif; max-width: 700px; margin: auto; color: #333; }",
        "    table { width: 100%; border-collapse: collapse; font-size: 14px; }",
        "    th, td { padding: 8px 10px; border: 1px solid #ddd; vertical-align: top; }",
        "    th { background: #f2f2f2; text-align: left; }",
        "  </style>",
        "</head>",
        "<body>",
        '  <h2 style="background:#003087;color:#fff;padding:12px 16px;'
        'border-radius:4px;margin:0 0 12px;">',
        "    &#127464;&#127462; CAD/USD Exchange&#8209;Rate Alert",
        "  </h2>",
        '  <p style="color:#888;font-size:12px;margin:0 0 16px;">Generated: ' + now + "</p>",
        '  <div style="background:' + overall_bg + ';border:1px solid #ccc;'
        'border-radius:4px;padding:12px 16px;margin-bottom:20px;">',
        "    " + score_txt + "<br/>",
        "    " + overall_icon + " " + overall_msg,
        "  </div>",
        "  <table>",
        "    <thead>",
        "      <tr>",
        "        <th>Indicator</th>",
        '        <th style="text-align:right;">Value</th>',
        "        <th>Signal</th>",
        "      </tr>",
        "    </thead>",
        "    <tbody>",
        "      " + row_a,
        "      " + row_b,
        "      " + row_c,
        "      " + row_d,
        "      " + row_e,
        "    </tbody>",
        "  </table>",
        '  <p style="font-size:11px;color:#aaa;margin-top:20px;">',
        "    Data: Bank of Canada Valet API &bull; Yahoo Finance.",
        "    Signals are informational only and do not constitute financial advice.",
        "  </p>",
        "</body>",
        "</html>",
    ]
    return "\n".join(parts)


In [ ]:
# ── SendGrid email sender ─────────────────────────────────────────────────────

def send_alert_email(html_content: str) -> bool:
    """
    Send the HTML report via SendGrid.
    Returns True on success, False on failure or missing API key.
    """
    if not SENDGRID_API_KEY:
        print("⚠️  SENDGRID_API_KEY not set – email NOT sent.")
        print("   Copy .env.example → .env and fill in your credentials.")
        return False

    message = Mail(
        from_email=FROM_EMAIL,
        to_emails=TO_EMAIL,
        subject=EMAIL_SUBJECT,
        html_content=html_content,
    )
    try:
        sg       = SendGridAPIClient(SENDGRID_API_KEY)
        response = sg.send(message)
        print(f"✅ Email sent successfully (HTTP {response.status_code})")
        return True
    except Exception as exc:
        print(f"❌ Failed to send email: {exc}")
        return False


In [ ]:
# ── Run all indicators and send the alert ─────────────────────────────────────
print("Fetching indicators…\n")

# A – CAD/USD
print("A: Fetching CAD/USD from Bank of Canada…")
cad_df = fetch_cad_usd(days=92)
cad    = analyze_cad_usd(cad_df)
print(f"   Spot: {cad['spot']}  |  30d avg: {cad['avg_30d']}  |  {cad['icon']} {cad['signal']}\n")

# B – DXY
print("B: Fetching USD Index (DXY) from Yahoo Finance…")
dxy = analyze_dxy()
print(f"   DXY: {dxy['current']}  |  5d slope: {dxy['5d_slope']}  |  {dxy['icon']} {dxy['signal']}\n")

# C – WTI
print("C: Fetching WTI Crude Oil from Yahoo Finance…")
wti = analyze_wti()
print(f"   WTI: ${wti['current']}  |  5d slope: {wti['5d_slope']}  |  {wti['icon']} {wti['signal']}\n")

# D – Yield spread
print("D: Fetching US–CA yield spread from Yahoo Finance…")
yld = analyze_yield_spread()
print(f"   US 10Y: {yld['us_10y']}  |  CA 10Y: {yld['ca_10y']}  |  Spread: {yld['spread']}")
print(f"   {yld['icon']} {yld['signal']}\n")

# E – VIX
print("E: Fetching VIX from Yahoo Finance…")
vix = analyze_vix()
print(f"   VIX: {vix['current']}  |  {vix['icon']} {vix['signal']}\n")

# Generate HTML report
print("Generating HTML report…")
html = generate_html_report(cad, dxy, wti, yld, vix)
print(f"   Report size: {len(html):,} bytes\n")

# Send email
print("Sending email via SendGrid…")
send_alert_email(html)

# Render report inline in the notebook
from IPython.display import HTML, display
display(HTML(html))
